# 05. Evaluación temporal de los escenarios

## 1. ¿Qué necesitamos comprobar después de construir los escenarios?

En la notebook 04 construimos escenarios conjuntos condicionados por grupos de VIX y verificamos su implementación numérica. Eso no demuestra que sus distribuciones representen mejor los movimientos posteriores.

Ahora compararemos pronósticos emitidos usando únicamente el pasado admisible con realizaciones posteriores. Repetiremos el ejercicio cronológicamente: una sola realización no valida una distribución; necesitamos una secuencia de pronósticos y resultados.

Referencia conceptual: Anton Vorobets, *Portfolio Construction and Risk Management*, sección 3.4, *Simulation Evaluation*, páginas impresas 49–53. El protocolo temporal y las métricas concretas serán decisiones de este proyecto.

## Alcance de esta notebook

- Compararemos remuestreo uniforme, exponencial y condicionado por VIX.
- Evaluaremos horizontes de una y cinco sesiones.
- En cada fecha, todos los métodos usarán la misma muestra admisible.
- Cortes, momentos y probabilidades se estimarán con el pasado de esa fecha; no se reutilizarán los ajustes de toda la muestra de 04.
- Examinaremos calidad marginal, dependencia conjunta y aspectos temporales.
- La evaluación es retrospectiva sobre datos ya explorados, no un holdout intacto ni una reconstrucción point-in-time completa.
- El objetivo final es decidir qué generador llevar a una posterior construcción de cartera candidata, o concluir que no hay evidencia suficiente para preferir el condicionado.
- No se evaluará rentabilidad de una estrategia ni se optimizarán carteras en 05.

La siguiente pregunta permanece abierta: **¿En qué momento emitimos cada pronóstico y qué información puede utilizar?**

## 2. ¿Cuándo emitimos el pronóstico y qué información utilizamos?

Antes de comparar modelos debemos separar la información utilizada para ajustarlos del intervalo cuyo retorno evaluaremos.

La convención acordada usa $t$ para una sesión del calendario de acciones publicado. El origen del pronóstico se sitúa inmediatamente antes del cierre de $t$. Usamos retornos históricos fechados hasta $t-1$ y el estado inicial utiliza el cierre VIX de $t-1$, conforme a nuestra convención de disponibilidad. No usamos el retorno de $t$ ni su cierre VIX para ajustar o inicializar el modelo.

| Elemento | Fecha o intervalo |
|---|---|
| Última fecha admisible de retornos de ajuste | $t-1$ |
| Fecha fuente del VIX inicial | $t-1$ |
| Precio de referencia del resultado | Cierre de $t$ |
| Primer retorno evaluado | Fila $t+1$ |
| Último retorno evaluado para $H=5$ | Fila $t+5$ |

Los resultados a evaluar son: para $H=1$, el retorno entre el cierre de $t$ y el cierre de $t+1$; para $H=5$, el retorno acumulado entre el cierre de $t$ y el cierre de $t+5$. Los índices representan sesiones, no días calendario.

El cierre de $t$ sólo se utiliza posteriormente para medir el resultado realizado; no es una entrada del pronóstico. Para la fila $t+1$, el VIX de $t-1$ respeta el rezago $x-2$ de la notebook 04. En los triples históricos de ajuste, tanto los retornos como sus estados deben respetar el límite de información establecido. Cortes, momentos crisp y probabilidades se reconstruirán exclusivamente con la muestra de ajuste de cada origen: no reutilizaremos $Q$, cortes ni momentos calculados sobre toda la muestra de 04.

La disponibilidad del VIX es una convención sustentada en la comunicación del proveedor, no una comprobación histórica point-in-time. La disponibilidad de las entradas sigue siendo un supuesto del experimento retrospectivo; no afirmamos disponer de archivos históricos con timestamps de publicación. Esta convención define un experimento de pronóstico, pero no presupone que podamos ejecutar operaciones al cierre de $t$.

**Ejemplo ilustrativo.** Antes del cierre del lunes, suponiendo sesiones habituales, usaríamos retornos de ajuste hasta el viernes y VIX inicial del viernes. El primer resultado sería el retorno del cierre del lunes al cierre del martes. No se selecciona aquí una fecha real.

Procedencia: la evaluación temporal se relaciona con la sección 3.4 del libro. El momento de emisión y el rezago son decisiones de este proyecto, no reglas atribuidas al autor.

La próxima pregunta es: **¿Cuánta historia utilizaremos para ajustar cada pronóstico y desde cuándo comenzará la evaluación?**

## 3. ¿Cuánta historia utilizamos y cuándo podemos evaluar?

Usaremos una ventana expansiva sobre las observaciones alineadas de 04, con mínimo inicial $M=504$. Cada origen incorpora observaciones nuevas admisibles. La semivida queda fija en $\tau=1102$ sesiones; no se recalcula como mitad de ventana. $M$ es una decisión inicial del proyecto, no un óptimo ni una regla del libro.

Este bloque construye fechas del experimento, no pronósticos: ajuste hasta $t-1$, VIX inicial de $t-1$, y resultados fechados $t+1$ hasta $t+H$, con $H\in\{1,5\}$. El retorno de $t$ no entra al ajuste ni al resultado.

In [1]:
from pathlib import Path
import json, hashlib, site, sys
import numpy as np
def root():
 for p in (Path.cwd(),*Path.cwd().parents):
  for c in (p,p/'scenario-portfolio-lab',p/'data-science-portfolio'/'scenario-portfolio-lab'):
   if (c/'config'/'universe.csv').is_file() and (c/'src'/'scenario_portfolio_lab').is_dir(): return c
 raise RuntimeError('Run from complete scenario-portfolio-lab repository.')
ROOT=root(); sp=ROOT/'.venv'/'Lib'/'site-packages'
if sp.is_dir():site.addsitedir(str(sp))
import polars as pl
RUN=ROOT/'data'/'published'/'cut_2026-10-09_run_2026-10-11_full_window_final'; AROOT=ROOT/'data'/'published'/'vix_state_alignment_2026-10-31'; VROOT=ROOT/'data'/'raw'/'vix_index_access_2026-10-29'
prices=pl.read_parquet(RUN/'gold'/'adjusted_close_matrix.parquet'); returns=pl.read_parquet(RUN/'gold'/'eligible_simple_total_return_matrix.parquet'); eligible=pl.read_csv(RUN/'reports'/'eligible_universe.csv').filter(pl.col('universe_status')=='ELIGIBLE'); audit=pl.read_parquet(AROOT/'vix_return_alignment.parquet').filter(pl.col('alignment_status')=='ALIGNED')
raw=VROOT/'raw'/'VIX_INDEX_DATA_daily.json'; manifest=json.loads((VROOT/'manifest.json').read_text(encoding='utf-8')); assert hashlib.sha256(raw.read_bytes()).hexdigest()==manifest['raw_sha256']
from datetime import date
if prices.columns[0] != returns.columns[0] or prices.columns[0] != 'session_date': raise AssertionError('Expected session_date as the temporal field in both matrices.')
if returns.columns[1:] != eligible['symbol'].to_list() or len(set(eligible['symbol'].to_list())) != 125: raise AssertionError('Eligible catalog identifiers do not match the matrix columns in order.')
vix={date.fromisoformat(x['date']):float(x['close']) for x in json.loads(raw.read_text(encoding='utf-8'))['data']}; cal=prices['session_date'].to_list(); aligned=audit['return_date'].to_list(); M,TAU,HORIZONS=504,1102,(1,5); retset=set(returns['session_date'].to_list())
tables={}
for H in HORIZONS:
 rows=[]
 for i in range(1,len(cal)):
  t,prev=cal[i],cal[i-1]; fit=[d for d in aligned if d<=prev]; future=cal[i+1:i+H+1]; reason='ELIGIBLE'
  if len(fit)<M:reason='INSUFFICIENT_ALIGNED_HISTORY'
  elif prev not in vix:reason='MISSING_INITIAL_VIX'
  elif len(future)!=H or any(d not in retset for d in future):reason='INCOMPLETE_FUTURE_HORIZON'
  rows.append({'H':H,'origin_date':t,'fit_first_return_date':fit[0] if fit else None,'fit_last_return_date':fit[-1] if fit else None,'fit_observations':len(fit),'initial_vix_date':prev,'initial_vix_close':vix.get(prev),'first_evaluated_return_date':future[0] if future else None,'last_evaluated_return_date':future[-1] if future else None,'eligible':reason=='ELIGIBLE','reason':reason})
 tables[H]=pl.DataFrame(rows)
eligible_tables={H:tables[H].filter(pl.col('eligible')) for H in HORIZONS}
controls={'calendar_unique_increasing':cal==sorted(cal) and len(cal)==len(set(cal)),'symbols_match':returns.columns[1:]==eligible['symbol'].to_list(),'expanding':all(np.all(np.diff(eligible_tables[H]['fit_observations'].to_numpy())==1) for H in HORIZONS),'fit_before_origin':all(all(r['fit_last_return_date']<r['origin_date'] and r['initial_vix_date']<r['origin_date'] for r in eligible_tables[H].to_dicts()) for H in HORIZONS),'future_complete':all(all(r['last_evaluated_return_date']>r['origin_date'] for r in eligible_tables[H].to_dicts()) for H in HORIZONS)}
if not all(controls.values()):raise AssertionError([k for k,v in controls.items() if not v])
display(pl.DataFrame({'configuration':['M','tau','symbols','calendar sessions','aligned observations'],'value':[M,TAU,len(returns.columns)-1,len(cal),len(aligned)]}))
for H in HORIZONS:display(eligible_tables[H].head(3));display(eligible_tables[H].tail(3));display(tables[H].group_by('reason').len().sort('reason'))
display(pl.DataFrame({'control':list(controls),'result':['PASS' if v else 'FAIL' for v in controls.values()]}));print('common origins',len(set(eligible_tables[1]['origin_date'])&set(eligible_tables[5]['origin_date'])))


configuration,value
str,i64
"""M""",504
"""tau""",1102
"""symbols""",125
"""calendar sessions""",2205
"""aligned observations""",2203


H,origin_date,fit_first_return_date,fit_last_return_date,fit_observations,initial_vix_date,initial_vix_close,first_evaluated_return_date,last_evaluated_return_date,eligible,reason
i64,date,date,date,i64,date,f64,date,date,bool,str
1,2020-01-07,2018-01-04,2020-01-06,504,2020-01-06,13.85,2020-01-08,2020-01-08,true,"""ELIGIBLE"""
1,2020-01-08,2018-01-04,2020-01-07,505,2020-01-07,13.79,2020-01-09,2020-01-09,true,"""ELIGIBLE"""
1,2020-01-09,2018-01-04,2020-01-08,506,2020-01-08,13.45,2020-01-10,2020-01-10,true,"""ELIGIBLE"""


H,origin_date,fit_first_return_date,fit_last_return_date,fit_observations,initial_vix_date,initial_vix_close,first_evaluated_return_date,last_evaluated_return_date,eligible,reason
i64,date,date,date,i64,date,f64,date,date,bool,str
1,2026-10-06,2018-01-04,2026-10-05,2199,2026-10-05,15.52,2026-10-07,2026-10-07,true,"""ELIGIBLE"""
1,2026-10-07,2018-01-04,2026-10-06,2200,2026-10-06,15.01,2026-10-08,2026-10-08,true,"""ELIGIBLE"""
1,2026-10-08,2018-01-04,2026-10-07,2201,2026-10-07,15.08,2026-10-09,2026-10-09,true,"""ELIGIBLE"""


reason,len
str,u32
"""ELIGIBLE""",1698
"""INCOMPLETE_FUTURE_HORIZON""",1
"""INSUFFICIENT_ALIGNED_HISTORY""",505


H,origin_date,fit_first_return_date,fit_last_return_date,fit_observations,initial_vix_date,initial_vix_close,first_evaluated_return_date,last_evaluated_return_date,eligible,reason
i64,date,date,date,i64,date,f64,date,date,bool,str
5,2020-01-07,2018-01-04,2020-01-06,504,2020-01-06,13.85,2020-01-08,2020-01-14,true,"""ELIGIBLE"""
5,2020-01-08,2018-01-04,2020-01-07,505,2020-01-07,13.79,2020-01-09,2020-01-15,true,"""ELIGIBLE"""
5,2020-01-09,2018-01-04,2020-01-08,506,2020-01-08,13.45,2020-01-10,2020-01-16,true,"""ELIGIBLE"""


H,origin_date,fit_first_return_date,fit_last_return_date,fit_observations,initial_vix_date,initial_vix_close,first_evaluated_return_date,last_evaluated_return_date,eligible,reason
i64,date,date,date,i64,date,f64,date,date,bool,str
5,2026-09-30,2018-01-04,2026-09-29,2195,2026-09-29,16.04,2026-10-01,2026-10-07,true,"""ELIGIBLE"""
5,2026-10-01,2018-01-04,2026-09-30,2196,2026-09-30,16.34,2026-10-02,2026-10-08,true,"""ELIGIBLE"""
5,2026-10-02,2018-01-04,2026-10-01,2197,2026-10-01,16.39,2026-10-05,2026-10-09,true,"""ELIGIBLE"""


reason,len
str,u32
"""ELIGIBLE""",1694
"""INCOMPLETE_FUTURE_HORIZON""",5
"""INSUFFICIENT_ALIGNED_HISTORY""",505


control,result
str,str
"""calendar_unique_increasing""","""PASS"""
"""symbols_match""","""PASS"""
"""expanding""","""PASS"""
"""fit_before_origin""","""PASS"""
"""future_complete""","""PASS"""


common origins 1694


## 4. ¿Cómo evaluar intervalos sin premiar que sean demasiado amplios?

Para un intervalo central nominal de 90%, definimos $L=q_{05}$, $U=q_{95}$ y $\alpha=0.10$. La **cobertura observada** es la proporción de realizaciones dentro de $[L,U]$; los incumplimientos inferior y superior ocurren cuando, respectivamente, $y<L$ y $y>U$. La **amplitud** es $U-L$.

Una cobertura elevada puede obtenerse con intervalos poco informativos si son demasiado amplios. Por ello utilizaremos el *interval score*:

$$IS_{\alpha}(L,U;y)=(U-L)+\frac{2}{\alpha}(L-y)\,\mathbf{1}\{y<L\}+\frac{2}{\alpha}(y-U)\,\mathbf{1}\{y>U\}.$$

Una puntuación menor es mejor: combina amplitud con una penalización proporcional a la distancia cuando la realización queda fuera del intervalo. Evalúa únicamente los dos cuantiles del intervalo, no toda la distribución pronosticada ni su dependencia conjunta. Además, la naturaleza discreta de nuestros pronósticos impide exigir cobertura exactamente igual a 90%.

**Procedencia.** Vorobets, sección 3.4, orienta la evaluación general. El *interval score* se toma de Gneiting y Raftery (2007), *Strictly Proper Scoring Rules, Prediction, and Estimation*, [DOI](https://doi.org/10.1198/016214506000001437). El intervalo nominal de 90% es una decisión de este proyecto.

**Ejemplo didáctico hipotético.** Consideremos A: $[-0.02,0.02]$ y B: $[-0.05,0.05]$. Para $y=0.01$, los scores son A=0.04 y B=0.10; para $y=0.03$, son A=0.24 y B=0.10. Los cálculos usan retornos decimales. Si se presentan como porcentajes, la puntuación reescalada se expresa en puntos porcentuales, no como rentabilidad de una inversión. Estos dos casos no se promedian ni permiten concluir nada sobre modelos reales.

Más adelante compararemos puntuaciones sobre las mismas fechas, instrumento y horizonte. No agregaremos sin más instrumentos de escalas distintas ni trataremos una diferencia media menor como evidencia estadística suficiente. La incertidumbre de las diferencias y el solapamiento de $H=5$ se abordarán posteriormente.

In [2]:
INTERVAL_ALPHA = 0.10
INTERVAL_ABSOLUTE_TOLERANCE = 1e-12
INTERVAL_RELATIVE_TOLERANCE = 1e-12

def interval_score(lower, upper, observed, *, alpha=INTERVAL_ALPHA):
    """Vectorized interval score with exact-shape inputs to prevent accidental broadcasting."""
    try:
        alpha = float(alpha)
    except (TypeError, ValueError) as exc:
        raise ValueError('alpha must be a finite scalar strictly between zero and one.') from exc
    if not np.isfinite(alpha) or not (0.0 < alpha < 1.0):
        raise ValueError('alpha must be strictly between zero and one.')
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    observed = np.asarray(observed, dtype=float)
    if lower.shape != upper.shape or lower.shape != observed.shape:
        raise ValueError('lower, upper, and observed must have identical shapes; broadcasting is not allowed.')
    if not (np.isfinite(lower).all() and np.isfinite(upper).all() and np.isfinite(observed).all()):
        raise ValueError('lower, upper, and observed must be finite.')
    if np.any(lower > upper):
        raise ValueError('lower must be less than or equal to upper elementwise.')
    return (upper - lower
            + (2.0 / alpha) * (lower - observed) * (observed < lower)
            + (2.0 / alpha) * (observed - upper) * (observed > upper))

def scalar_interval_score(lower, upper, observed, alpha):
    width = upper - lower
    if observed < lower:
        return width + (2.0 / alpha) * (lower - observed)
    if observed > upper:
        return width + (2.0 / alpha) * (observed - upper)
    return width

example_lower = np.array([-0.02, -0.05, -0.02, -0.05])
example_upper = np.array([ 0.02,  0.05,  0.02,  0.05])
example_observed = np.array([0.01, 0.01, 0.03, 0.03])
example_scores = interval_score(example_lower, example_upper, example_observed)
expected_scores = np.array([0.04, 0.10, 0.24, 0.10])

inside_score = interval_score(np.array([-0.02]), np.array([0.02]), np.array([0.01]))[0]
at_lower_score = interval_score(np.array([-0.02]), np.array([0.02]), np.array([-0.02]))[0]
at_upper_score = interval_score(np.array([-0.02]), np.array([0.02]), np.array([0.02]))[0]
below_score = interval_score(np.array([-0.02]), np.array([0.02]), np.array([-0.03]))[0]
above_score = interval_score(np.array([-0.02]), np.array([0.02]), np.array([0.03]))[0]
independent_scores = np.array([scalar_interval_score(l, u, y, INTERVAL_ALPHA) for l, u, y in zip(example_lower, example_upper, example_observed)])

invalid_cases = {
    'alpha_zero': lambda: interval_score(np.array([0.0]), np.array([1.0]), np.array([0.5]), alpha=0.0),
    'nonfinite_input': lambda: interval_score(np.array([np.nan]), np.array([1.0]), np.array([0.5])),
    'reversed_bounds': lambda: interval_score(np.array([1.0]), np.array([0.0]), np.array([0.5])),
    'shape_mismatch': lambda: interval_score(np.zeros(2), np.zeros(3), np.zeros(2)),
}
invalid_rejected = {}
for name, call in invalid_cases.items():
    try:
        call()
    except ValueError:
        invalid_rejected[name] = True
    else:
        invalid_rejected[name] = False

checks = {
    'four_hypothetical_scores': np.allclose(example_scores, expected_scores, atol=INTERVAL_ABSOLUTE_TOLERANCE, rtol=INTERVAL_RELATIVE_TOLERANCE),
    'inside_equals_width': np.isclose(inside_score, 0.04, atol=INTERVAL_ABSOLUTE_TOLERANCE, rtol=INTERVAL_RELATIVE_TOLERANCE),
    'boundaries_have_no_extra_penalty': np.isclose(at_lower_score, 0.04, atol=INTERVAL_ABSOLUTE_TOLERANCE, rtol=INTERVAL_RELATIVE_TOLERANCE) and np.isclose(at_upper_score, 0.04, atol=INTERVAL_ABSOLUTE_TOLERANCE, rtol=INTERVAL_RELATIVE_TOLERANCE),
    'below_penalty': np.isclose(below_score, 0.24, atol=INTERVAL_ABSOLUTE_TOLERANCE, rtol=INTERVAL_RELATIVE_TOLERANCE),
    'above_penalty': np.isclose(above_score, 0.24, atol=INTERVAL_ABSOLUTE_TOLERANCE, rtol=INTERVAL_RELATIVE_TOLERANCE),
    'scalar_concordance': np.allclose(example_scores, independent_scores, atol=INTERVAL_ABSOLUTE_TOLERANCE, rtol=INTERVAL_RELATIVE_TOLERANCE),
    'invalid_inputs_rejected': all(invalid_rejected.values()),
}
if not all(checks.values()):
    raise AssertionError([name for name, passed in checks.items() if not passed])

display(pl.DataFrame({
    'interval': ['A', 'B', 'A', 'B'],
    'observed_decimal': example_observed,
    'interval_score_decimal': example_scores,
    'interval_score_percentage_points': example_scores * 100.0,
}))
display(pl.DataFrame({'control': list(checks), 'result': ['PASS' if passed else 'FAIL' for passed in checks.values()]}))
print(f'alpha={INTERVAL_ALPHA:.2f}; tolerances: absolute={INTERVAL_ABSOLUTE_TOLERANCE:g}, relative={INTERVAL_RELATIVE_TOLERANCE:g}')


interval,observed_decimal,interval_score_decimal,interval_score_percentage_points
str,f64,f64,f64
"""A""",0.01,0.04,4.0
"""B""",0.01,0.1,10.0
"""A""",0.03,0.24,24.0
"""B""",0.03,0.1,10.0


control,result
str,str
"""four_hypothetical_scores""","""PASS"""
"""inside_equals_width""","""PASS"""
"""boundaries_have_no_extra_penal…","""PASS"""
"""below_penalty""","""PASS"""
"""above_penalty""","""PASS"""
"""scalar_concordance""","""PASS"""
"""invalid_inputs_rejected""","""PASS"""


alpha=0.10; tolerances: absolute=1e-12, relative=1e-12
